In [1]:
import torch
import torch.nn as nn
import numpy as np
import nibabel as nib
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import os
from typing import List, Tuple, Dict
import random
import pandas as pd
import time
from datetime import timedelta
import torch.multiprocessing as mp
import json

In [2]:
class CSVSubjectNifti3DDataset(Dataset):
    def __init__(self, subject_ids):
       
        self.subject_ids = subject_ids
        # Collect all file pairs for the specified subjects and file indices
        self.metadata = [] # Store (subject_id, file_index) for each sample
        self.file_pairs = []  
        for subject_id in subject_ids:
            
            for i in range(1, 6):
                cond_file = f"C:/Dataset For training/Conductivity maps/{subject_id}/conductivity_map_{i}.nii.gz"
                volt_file = f"C:/Dataset For training/voltage/{subject_id}/voltage_map_{i}.nii.gz"

                #cond_file = f"D:/Results/extrapolation/conductivity/{subject_id}/conductivity_map_{i}.nii.gz"
                #volt_file = f"D:/Results/extrapolation/voltage/{subject_id}/voltage_map_{i}.nii.gz"
                
                self.file_pairs.append((cond_file , volt_file))
                self.metadata.append((subject_id, i))
        
        print(f"Found {len(self.file_pairs)} conductivity-voltage pairs for {len(subject_ids)} subjects")
        
    def __len__(self):
        return len(self.file_pairs)
    
    def __getitem__(self, idx):
        conductivity_file, voltage_file = self.file_pairs[idx]
        conductivity_img = nib.load(conductivity_file)
        conductivity_data = conductivity_img.get_fdata().astype(np.float32)
        mask = conductivity_data > 100 #places of electrode have hight conductivity
        conductivity_data[mask] = 2 #make it 2 to avoid affect training process
        voltage_img = nib.load(voltage_file)
        voltage_data = voltage_img.get_fdata().astype(np.float32)
        
        conductivity_tensor = torch.from_numpy(conductivity_data)
        voltage_tensor = torch.from_numpy(voltage_data)
        
        # Create boundary mask (where voltage is not zero)
        boundary_mask = (voltage_tensor != 0)
        
        subject_id, file_index = self.metadata[idx]
        
        
        
        return {
            'conductivity_map': conductivity_tensor,
            'voltage_bc': voltage_tensor,
            'boundary_mask': boundary_mask,
            'subject_id': subject_id,
            'file_index': file_index,
        }



In [3]:
class DoubleConv3D(nn.Module):
    """(3D convolution => LeakyReLU) * 2 with same padding"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        self.double_conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(negative_slope=0.01)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down3D(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool3d(2),
            DoubleConv3D(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up3D(nn.Module):
    """Upscaling then double conv with skip connection"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = DoubleConv3D(out_channels * 2, out_channels) # we can put in_channel in first input

    def forward(self, x, skip):
        x = self.up(x)

        # Handle size mismatches automatically
        diffZ = skip.size()[2] - x.size()[2]
        diffY = skip.size()[3] - x.size()[3]
        diffX = skip.size()[4] - x.size()[4]
        
        # Apply padding if upsampled tensor is smaller
        x = nn.functional.pad(x, [diffX // 2, diffX - diffX // 2,
                      diffY // 2, diffY - diffY // 2,
                      diffZ // 2, diffZ - diffZ // 2])
        
        # Input and skip connection now have the same size
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class Surrogate3DPINN(nn.Module):
    """3D U-Net with padding to maintain spatial dimensions"""
    def __init__(self):
        super(Surrogate3DPINN, self).__init__()
        
        # Encoder
        self.inc = DoubleConv3D(2, 8)
        self.down1 = Down3D(8, 16)
        self.down2 = Down3D(16, 32)
        self.down3 = Down3D(32, 64)
        self.down4 = Down3D(64, 128)
        
        # Decoder
        self.up1 = Up3D(128, 64)
        self.up2 = Up3D(64, 32)
        self.up3 = Up3D(32, 16)
        self.up4 = Up3D(16, 8)
        
        # Output
        self.outc = nn.Conv3d(8, 1, kernel_size=1)

    def forward(self, conductivity, voltage_bc):

        x = torch.cat([conductivity, voltage_bc], dim=1)  # [batch, 2, D, H, W]
        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        # Decoder
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        
        return self.outc(x)

In [4]:
def finite_difference_gradients_3d(potential: torch.Tensor, dx: float = 0.001) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute 3D gradients using finite differences"""
    # Initialize gradient tensors
    grad_x = torch.zeros_like(potential)
    grad_y = torch.zeros_like(potential)
    grad_z = torch.zeros_like(potential)
    
    # x-gradient (central difference for interior, forward/backward for boundaries)
    grad_x[:, :, 1:-1, :, :] = (potential[:, :, 2:, :, :] - potential[:, :, :-2, :, :]) / (2 * dx)
    grad_x[:, :, 0, :, :] = (potential[:, :, 1, :, :] - potential[:, :, 0, :, :]) / dx
    grad_x[:, :, -1, :, :] = (potential[:, :, -1, :, :] - potential[:, :, -2, :, :]) / dx
    
    # y-gradient
    grad_y[:, :, :, 1:-1, :] = (potential[:, :, :, 2:, :] - potential[:, :, :, :-2, :]) / (2 * dx)
    grad_y[:, :, :, 0, :] = (potential[:, :, :, 1, :] - potential[:, :, :, 0, :]) / dx
    grad_y[:, :, :, -1, :] = (potential[:, :, :, -1, :] - potential[:, :, :, -2, :]) / dx
    
    # z-gradient
    grad_z[:, :, :, :, 1:-1] = (potential[:, :, :, :, 2:] - potential[:, :, :, :, :-2]) / (2 * dx)
    grad_z[:, :, :, :, 0] = (potential[:, :, :, :, 1] - potential[:, :, :, :, 0]) / dx
    grad_z[:, :, :, :, -1] = (potential[:, :, :, :, -1] - potential[:, :, :, :, -2]) / dx
    
    return grad_x, grad_y, grad_z

def physics_loss_3d(potential: torch.Tensor, conductivity_map: torch.Tensor, dx: float = 0.001) -> torch.Tensor:
    """Compute ∇·(σ∇φ) = 0 residual in 3D using finite differences"""
    # Compute gradients ∇φ
    grad_x, grad_y, grad_z = finite_difference_gradients_3d(potential, dx)
    
    # Compute σ∇φ
    sigma_grad_x = conductivity_map * grad_x
    sigma_grad_y = conductivity_map * grad_y
    sigma_grad_z = conductivity_map * grad_z
    
    # Compute divergence ∇·(σ∇φ)
    div_x, _, _ = finite_difference_gradients_3d(sigma_grad_x, dx)
    _, div_y, _ = finite_difference_gradients_3d(sigma_grad_y, dx)
    _, _, div_z = finite_difference_gradients_3d(sigma_grad_z, dx)
    
    divergence = div_x + div_y + div_z
    #divergence = torch.abs(divergence)
    # Physics residual (should be zero)
    physics_residual = torch.mean(divergence**2)
    #physics_residual = torch.mean(divergence)
    return physics_residual

def energy_loss_3d(potential, conductivity_map, dx=0.001):
    # Compute energy density: ½ σ |∇φ|²
    grad_x, grad_y, grad_z = finite_difference_gradients_3d(potential, dx)
    
    # |∇φ|² = (∂φ/∂x)² + (∂φ/∂y)² + (∂φ/∂z)²
    grad_squared = grad_x**2 + grad_y**2 + grad_z**2
    
    # Energy density: ½ σ |∇φ|²
    energy_density = 0.5 * conductivity_map * grad_squared
    
    # Total energy (integrate over volume)
    total_energy = torch.mean(energy_density) 
    
    return total_energy

In [5]:
def load_and_split_subjects(csv_file_path , train_ratio = 0.8, random_state = 42):
    df = pd.read_csv(csv_file_path)
    subject_ids = df['Subject ID']
    
    # Set a seed for reproducibility
    np.random.seed(random_state)
    # Shuffle the list and split it
    train_ids, test_ids = np.split(np.random.permutation(subject_ids), [int(train_ratio * len(subject_ids))])
    return train_ids, test_ids

In [7]:
def train_model_3d(model: nn.Module, train_loader: DataLoader, 
                  optimizer: torch.optim.Optimizer, device: torch.device, num_epochs: int = 1000):
    """Train the 3D PINN model with validation and loss tracking"""
    model.train()
    
    # Initialize lists to store losses
    train_physics_losses = []
    epochs_list = []
    
    # Create directory for saving loss data
    save_dir = "D:/Results/hard enforcement"

    start_time = time.time()
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        epoch_train_physics = 0.0
        
        for batch in train_loader:
            # Move data to device
            voltage_bc = batch['voltage_bc'].to(device).unsqueeze(1)     # [B, 1, D, H, W]
            
            conductivity = batch['conductivity_map'].to(device).unsqueeze(1)  # [B, 1, D, H, W]
            boundary_mask = batch['boundary_mask'].to(device).unsqueeze(1)   # [B, 1, D, H, W]

            
            # Forward pass
            potential_pred = model(conductivity, voltage_bc)
            potential_pred[boundary_mask] = voltage_bc[boundary_mask]
            # Compute losses
            
            p_loss = physics_loss_3d(potential_pred, conductivity, dx=0.001) * 1e-18
            
           
            optimizer.zero_grad()
            p_loss.backward()
            optimizer.step()
            # Accumulate losses
            epoch_train_physics += p_loss
            
        
        # Calculate averages
        avg_train_physics = (epoch_train_physics / len(train_loader)).item()
       
        # Store losses for plotting
        train_physics_losses.append(avg_train_physics)
       
        epochs_list.append(epoch)
        
        # Print statistics
        print(f'Epoch {epoch:4d}/{num_epochs}')
        print(f'  Train - Physics: {avg_train_physics:.6f}')
       
        
    #calclate training time
    total_time = time.time() - start_time
    total_time_str = str(timedelta(seconds=int(total_time)))
    
    # Save all loss data to files
    loss_data = {
        'epochs': epochs_list,
        'train_physics_losses': train_physics_losses,
        'training time': total_time_str}
        
    
    
    # Save as JSON
    with open(os.path.join(save_dir, 'loss_data4.json'), 'w') as f:
        json.dump(loss_data, f, indent=4)

    print(f"All loss data saved to directory: {save_dir}")
    
    return model, optimizer, loss_data

In [6]:
def save_model(model, optimizer, path):
    torch.save({
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
    }, path)

def load_model(model, optimizer, path):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])

In [ ]:
if __name__ == '__main__':
    mp.set_start_method('spawn', force=True)

    # Configuration
    csv_file_path = "E:/Datasets/Stage 2/add electrode folder/All Subjects .csv"
    batch_size = 1
    num_epochs = 40
    learning_rate = 1e-4
    train_ratio = 0.8
    
    
    # Device configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load and split subject IDs
    print("Loading and splitting subject IDs from CSV...")
    train_subjects, test_subjects = load_and_split_subjects(csv_file_path, train_ratio)
    
    # Set random seeds
    torch.manual_seed(42)
    #torch.manual_seed(12345)
    #torch.manual_seed(67890)
    
    # Create datasets
    train_dataset = CSVSubjectNifti3DDataset(train_subjects)
    test_dataset = CSVSubjectNifti3DDataset(test_subjects)
    
    print(f"Dataset sizes: Train={len(train_dataset)}, Test={len(test_dataset)}")
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True , num_workers = 2, prefetch_factor = 2 , pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )
    
    
    model = Surrogate3DPINN().to(device)
    model = nn.DataParallel(model)
    
    # Print model parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")
    
    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    #load_model(model , optimizer, "D:/Results/train3.pth")
    # Train the model
    print("Starting 3D training with U-Net architecture...")
    model, optimizer, train_losses = train_model_3d(model, train_loader, optimizer, device, num_epochs)
    save_model(model, optimizer,"D:/Results/hard enforcement/train1.pth" )
    print("3D U-Net model saved")

In [7]:
def test_model_3d(model: nn.Module, test_loader: DataLoader, device: torch.device):
    """Test the trained model on test set"""
    model.eval()
    
    test_physics_loss = 0.0
    test_energy_loss = 0.0
    
    with torch.no_grad():
        for batch in test_loader:
            conductivity = batch['conductivity_map'].unsqueeze(1).to(device)
            voltage_bc = batch['voltage_bc'].unsqueeze(1).to(device)
            boundary_mask = batch['boundary_mask'].unsqueeze(1).to(device)
                        
            potential_pred = model(conductivity, voltage_bc)
            potential_pred = torch.where(boundary_mask, voltage_bc, potential_pred)
            # Compute losses
            
            p_loss = physics_loss_3d(potential_pred, conductivity, dx=0.001)
            e_loss = energy_loss_3d(potential_pred, conductivity, dx=0.001)
            
            test_physics_loss += p_loss
            test_energy_loss += e_loss
    
    avg_test_physics = (test_physics_loss / len(test_loader)).item()
    avg_test_energy = (test_energy_loss / len(test_loader)).item()
    
    print(f'\nTest Results:')
    print(f'Physics Loss: {avg_test_physics:.6f}')
    print(f'Energy Loss: {avg_test_energy:.6f}')
    
    return avg_test_physics, avg_test_energy

In [10]:
csv_file_path = "E:/Datasets/Stage 2/add electrode folder/All Subjects .csv"
train_ratio = 0.8
learning_rate = 1e-4
device = torch.device('cuda')

# Load and split subject IDs
print("Loading and splitting subject IDs from CSV...")
train_subjects, test_subjects = load_and_split_subjects(csv_file_path, train_ratio)


train_dataset = CSVSubjectNifti3DDataset(train_subjects)
test_dataset = CSVSubjectNifti3DDataset(test_subjects)

print(f"Dataset sizes: Train={len(train_dataset)}, Test={len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset,batch_size = 1, shuffle = False )
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )

print(f"Dataset sizes: Test={len(test_dataset)}")

# Create dataloader
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )


model = Surrogate3DPINN().to(device)
model = nn.DataParallel(model)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
load_model(model , optimizer, "D:/Results/hard enforcement/train6.pth")

# Test
print("Test Data:")
test_model_3d(model, test_loader, device)
print("Train Data:")
test_model_3d(model, train_loader, device)


Loading and splitting subject IDs from CSV...
Found 405 conductivity-voltage pairs for 81 subjects
Found 105 conductivity-voltage pairs for 21 subjects
Dataset sizes: Train=405, Test=105
Dataset sizes: Test=105
Test Data:

Test Results:
Physics Loss: 1263.571777
Energy Loss: 564040768.000000
Train Data:

Test Results:
Physics Loss: 1.178398
Energy Loss: 2615793.500000


(1.1783980131149292, 2615793.5)

In [11]:
csv_file_path = "E:/Datasets/Stage 2/add electrode folder/All Subjects .csv"
train_ratio = 0.8
device = torch.device('cuda')

# Load and split subject IDs
print("Loading and splitting subject IDs from CSV...")
train_subjects, test_subjects = load_and_split_subjects(csv_file_path, train_ratio)

train_dataset = CSVSubjectNifti3DDataset(train_subjects)
test_dataset = CSVSubjectNifti3DDataset(test_subjects)

print(f"Dataset sizes: Train={len(train_dataset)}, Test={len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset,batch_size = 1, shuffle = False )
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )

#print(f"Dataset sizes: Test={len(test_dataset)}")

# Create dataloader


model = Surrogate3DPINN().to(device)
model = nn.DataParallel(model)

# Optimizer
path = "D:/Results/Energy_Loss/min_loss_model.pth"
state_dict = torch.load(path)
model.load_state_dict(state_dict)


# Test
print("Test Data:")
test_model_3d(model, test_loader, device)
print("Train Data:")
test_model_3d(model, train_loader, device)

Loading and splitting subject IDs from CSV...
Found 405 conductivity-voltage pairs for 81 subjects
Found 105 conductivity-voltage pairs for 21 subjects
Dataset sizes: Train=405, Test=105
Test Data:

Test Results:
Physics Loss: 1255.855591
Energy Loss: 556024384.000000
Train Data:

Test Results:
Physics Loss: 0.947699
Energy Loss: 467587.812500


(0.9476993680000305, 467587.8125)

In [8]:
csv_file_path = "E:/Datasets/Stage 2/add electrode folder/All Subjects .csv"
train_ratio = 0.8
learning_rate = 1e-4
device = torch.device('cuda')

# Load and split subject IDs
print("Loading and splitting subject IDs from CSV...")
train_subjects, test_subjects = load_and_split_subjects(csv_file_path, train_ratio)


train_dataset = CSVSubjectNifti3DDataset(train_subjects)
test_dataset = CSVSubjectNifti3DDataset(test_subjects)

print(f"Dataset sizes: Train={len(train_dataset)}, Test={len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset,batch_size = 1, shuffle = False )
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )

print(f"Dataset sizes: Test={len(test_dataset)}")

# Create dataloader
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )


model = Surrogate3DPINN().to(device)
model = nn.DataParallel(model)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
load_model(model , optimizer, "D:/Results/Energy loss 2/train3.pth")

# Test
print("Test Data:")
test_model_3d(model, test_loader, device)
print("Train Data:")
test_model_3d(model, train_loader, device)

Loading and splitting subject IDs from CSV...
Found 405 conductivity-voltage pairs for 81 subjects
Found 105 conductivity-voltage pairs for 21 subjects
Dataset sizes: Train=405, Test=105
Dataset sizes: Test=105
Test Data:

Test Results:
Physics Loss: 1643126.125000
Energy Loss: 3.960595
Train Data:

Test Results:
Physics Loss: 659793.875000
Energy Loss: 3.549582


(659793.875, 3.54958176612854)

In [10]:
csv_file_path = "E:/Datasets/Stage 2/add electrode folder/All Subjects .csv"
train_ratio = 0.8
learning_rate = 1e-4
device = torch.device('cuda')

# Load and split subject IDs
print("Loading and splitting subject IDs from CSV...")
train_subjects, test_subjects = load_and_split_subjects(csv_file_path, train_ratio)


train_dataset = CSVSubjectNifti3DDataset(train_subjects)
test_dataset = CSVSubjectNifti3DDataset(test_subjects)

print(f"Dataset sizes: Train={len(train_dataset)}, Test={len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset,batch_size = 1, shuffle = False )
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )

print(f"Dataset sizes: Test={len(test_dataset)}")

# Create dataloader
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )


model = Surrogate3DPINN().to(device)
model = nn.DataParallel(model)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
load_model(model , optimizer, "D:/Results/hard enforcement 2/train3.pth")

# Test
print("Test Data:")
test_model_3d(model, test_loader, device)
print("Train Data:")
test_model_3d(model, train_loader, device)

Loading and splitting subject IDs from CSV...
Found 405 conductivity-voltage pairs for 81 subjects
Found 105 conductivity-voltage pairs for 21 subjects
Dataset sizes: Train=405, Test=105
Dataset sizes: Test=105
Test Data:

Test Results:
Physics Loss: 1486756.000000
Energy Loss: 7.198119
Train Data:

Test Results:
Physics Loss: 637647.062500
Energy Loss: 6.742041


(637647.0625, 6.742041110992432)

In [14]:
## extrapolation conductivity
csv_file_path = "E:/Datasets/Stage 2/add electrode folder/All Subjects .csv"
train_ratio = 0.8
learning_rate = 1e-4
device = torch.device('cuda')

# Load and split subject IDs
print("Loading and splitting subject IDs from CSV...")
train_subjects, test_subjects = load_and_split_subjects(csv_file_path, train_ratio)


train_dataset = CSVSubjectNifti3DDataset(train_subjects)
test_dataset = CSVSubjectNifti3DDataset(test_subjects)

print(f"Dataset sizes: Train={len(train_dataset)}, Test={len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset,batch_size = 1, shuffle = False )
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )

print(f"Dataset sizes: Test={len(test_dataset)}")

# Create dataloader
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )


model = Surrogate3DPINN().to(device)
model = nn.DataParallel(model)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
load_model(model , optimizer, "D:/Results/Energy loss 2/train3.pth")

# Test
print("Test Data:")
test_model_3d(model, test_loader, device)
print("Train Data:")
test_model_3d(model, train_loader, device)

Loading and splitting subject IDs from CSV...
Found 324 conductivity-voltage pairs for 81 subjects
Found 84 conductivity-voltage pairs for 21 subjects
Dataset sizes: Train=324, Test=84
Dataset sizes: Test=84
Test Data:

Test Results:
Physics Loss: 1176319.500000
Energy Loss: 3.781050
Train Data:

Test Results:
Physics Loss: 809599.312500
Energy Loss: 3.644788


(809599.3125, 3.6447882652282715)

In [21]:
## abs error instead of mean
csv_file_path = "E:/Datasets/Stage 2/add electrode folder/All Subjects .csv"
train_ratio = 0.8
learning_rate = 1e-4
device = torch.device('cuda')

# Load and split subject IDs
print("Loading and splitting subject IDs from CSV...")
train_subjects, test_subjects = load_and_split_subjects(csv_file_path, train_ratio)


train_dataset = CSVSubjectNifti3DDataset(train_subjects)
test_dataset = CSVSubjectNifti3DDataset(test_subjects)

print(f"Dataset sizes: Train={len(train_dataset)}, Test={len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset,batch_size = 1, shuffle = False )
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )

print(f"Dataset sizes: Test={len(test_dataset)}")

# Create dataloader
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )


model = Surrogate3DPINN().to(device)
model = nn.DataParallel(model)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
load_model(model , optimizer, "D:/Results/Energy loss 2/train3.pth")

# Test
print("Test Data:")
test_model_3d(model, test_loader, device)
print("Train Data:")
test_model_3d(model, train_loader, device)

Loading and splitting subject IDs from CSV...
Found 405 conductivity-voltage pairs for 81 subjects
Found 105 conductivity-voltage pairs for 21 subjects
Dataset sizes: Train=405, Test=105
Dataset sizes: Test=105
Test Data:

Test Results:
Physics Loss: 176.119843
Energy Loss: 3.960595
Train Data:

Test Results:
Physics Loss: 164.029861
Energy Loss: 3.549582


(164.0298614501953, 3.54958176612854)

In [22]:
## abs error instead of mean
csv_file_path = "E:/Datasets/Stage 2/add electrode folder/All Subjects .csv"
train_ratio = 0.8
learning_rate = 1e-4
device = torch.device('cuda')

# Load and split subject IDs
print("Loading and splitting subject IDs from CSV...")
train_subjects, test_subjects = load_and_split_subjects(csv_file_path, train_ratio)


train_dataset = CSVSubjectNifti3DDataset(train_subjects)
test_dataset = CSVSubjectNifti3DDataset(test_subjects)

print(f"Dataset sizes: Train={len(train_dataset)}, Test={len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset,batch_size = 1, shuffle = False )
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )

print(f"Dataset sizes: Test={len(test_dataset)}")

# Create dataloader
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )


model = Surrogate3DPINN().to(device)
model = nn.DataParallel(model)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
load_model(model , optimizer, "D:/Results/hard enforcement 2/train3.pth")

# Test
print("Test Data:")
test_model_3d(model, test_loader, device)
print("Train Data:")
test_model_3d(model, train_loader, device)

Loading and splitting subject IDs from CSV...
Found 405 conductivity-voltage pairs for 81 subjects
Found 105 conductivity-voltage pairs for 21 subjects
Dataset sizes: Train=405, Test=105
Dataset sizes: Test=105
Test Data:

Test Results:
Physics Loss: 192.417603
Energy Loss: 7.198119
Train Data:

Test Results:
Physics Loss: 179.293533
Energy Loss: 6.742041


(179.2935333251953, 6.742041110992432)

In [8]:
csv_file_path = "E:/Datasets/Stage 2/add electrode folder/All Subjects .csv"
train_ratio = 0.8
learning_rate = 1e-4
device = torch.device('cuda')

# Load and split subject IDs
print("Loading and splitting subject IDs from CSV...")
train_subjects, test_subjects = load_and_split_subjects(csv_file_path, train_ratio)


train_dataset = CSVSubjectNifti3DDataset(train_subjects)
test_dataset = CSVSubjectNifti3DDataset(test_subjects)

print(f"Dataset sizes: Train={len(train_dataset)}, Test={len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset,batch_size = 1, shuffle = False )
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )

print(f"Dataset sizes: Test={len(test_dataset)}")

# Create dataloader
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )


model = Surrogate3DPINN().to(device)
model = nn.DataParallel(model)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
load_model(model , optimizer, "D:/Results/Energy loss 2/train2.pth")

# Test
print("Test Data:")
test_model_3d(model, test_loader, device)
print("Train Data:")
test_model_3d(model, train_loader, device)

Loading and splitting subject IDs from CSV...
Found 405 conductivity-voltage pairs for 81 subjects
Found 105 conductivity-voltage pairs for 21 subjects
Dataset sizes: Train=405, Test=105
Dataset sizes: Test=105
Test Data:

Test Results:
Physics Loss: 2527932.250000
Energy Loss: 4.383580
Train Data:

Test Results:
Physics Loss: 970687.625000
Energy Loss: 3.821416


(970687.625, 3.821416139602661)

In [9]:
csv_file_path = "E:/Datasets/Stage 2/add electrode folder/All Subjects .csv"
train_ratio = 0.8
learning_rate = 1e-4
device = torch.device('cuda')

# Load and split subject IDs
print("Loading and splitting subject IDs from CSV...")
train_subjects, test_subjects = load_and_split_subjects(csv_file_path, train_ratio)


train_dataset = CSVSubjectNifti3DDataset(train_subjects)
test_dataset = CSVSubjectNifti3DDataset(test_subjects)

print(f"Dataset sizes: Train={len(train_dataset)}, Test={len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset,batch_size = 1, shuffle = False )
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )

print(f"Dataset sizes: Test={len(test_dataset)}")

# Create dataloader
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )


model = Surrogate3DPINN().to(device)
model = nn.DataParallel(model)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
load_model(model , optimizer, "D:/Results/hard enforcement 2/train2.pth")

# Test
print("Test Data:")
test_model_3d(model, test_loader, device)
print("Train Data:")
test_model_3d(model, train_loader, device)

Loading and splitting subject IDs from CSV...
Found 405 conductivity-voltage pairs for 81 subjects
Found 105 conductivity-voltage pairs for 21 subjects
Dataset sizes: Train=405, Test=105
Dataset sizes: Test=105
Test Data:

Test Results:
Physics Loss: 2185589.250000
Energy Loss: 8.069111
Train Data:

Test Results:
Physics Loss: 890274.437500
Energy Loss: 7.329328


(890274.4375, 7.3293280601501465)

In [10]:
csv_file_path = "E:/Datasets/Stage 2/add electrode folder/All Subjects .csv"
train_ratio = 0.8
learning_rate = 1e-4
device = torch.device('cuda')

# Load and split subject IDs
print("Loading and splitting subject IDs from CSV...")
train_subjects, test_subjects = load_and_split_subjects(csv_file_path, train_ratio)


train_dataset = CSVSubjectNifti3DDataset(train_subjects)
test_dataset = CSVSubjectNifti3DDataset(test_subjects)

print(f"Dataset sizes: Train={len(train_dataset)}, Test={len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset,batch_size = 1, shuffle = False )
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )

print(f"Dataset sizes: Test={len(test_dataset)}")

# Create dataloader
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )


model = Surrogate3DPINN().to(device)
model = nn.DataParallel(model)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
load_model(model , optimizer, "D:/Results/Energy loss 2/train1.pth")

# Test
print("Test Data:")
test_model_3d(model, test_loader, device)
print("Train Data:")
test_model_3d(model, train_loader, device)

Loading and splitting subject IDs from CSV...
Found 405 conductivity-voltage pairs for 81 subjects
Found 105 conductivity-voltage pairs for 21 subjects
Dataset sizes: Train=405, Test=105
Dataset sizes: Test=105
Test Data:

Test Results:
Physics Loss: 4636024.000000
Energy Loss: 5.722029
Train Data:

Test Results:
Physics Loss: 1835917.000000
Energy Loss: 4.749595


(1835917.0, 4.7495951652526855)

In [11]:
csv_file_path = "E:/Datasets/Stage 2/add electrode folder/All Subjects .csv"
train_ratio = 0.8
learning_rate = 1e-4
device = torch.device('cuda')

# Load and split subject IDs
print("Loading and splitting subject IDs from CSV...")
train_subjects, test_subjects = load_and_split_subjects(csv_file_path, train_ratio)


train_dataset = CSVSubjectNifti3DDataset(train_subjects)
test_dataset = CSVSubjectNifti3DDataset(test_subjects)

print(f"Dataset sizes: Train={len(train_dataset)}, Test={len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset,batch_size = 1, shuffle = False )
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )

print(f"Dataset sizes: Test={len(test_dataset)}")

# Create dataloader
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False )


model = Surrogate3DPINN().to(device)
model = nn.DataParallel(model)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
load_model(model , optimizer, "D:/Results/hard enforcement 2/train1.pth")

# Test
print("Test Data:")
test_model_3d(model, test_loader, device)
print("Train Data:")
test_model_3d(model, train_loader, device)

Loading and splitting subject IDs from CSV...
Found 405 conductivity-voltage pairs for 81 subjects
Found 105 conductivity-voltage pairs for 21 subjects
Dataset sizes: Train=405, Test=105
Dataset sizes: Test=105
Test Data:

Test Results:
Physics Loss: 3761880.500000
Energy Loss: 9.189014
Train Data:

Test Results:
Physics Loss: 1348271.000000
Energy Loss: 8.076306


(1348271.0, 8.076306343078613)